# TD4 — Feature engineering EEG pour l'estimation de la charge cognitive

## Contexte

Ce TD s'inscrit dans le projet fil rouge basé sur le papier :

**Multimodal Brain-Computer Interface for In-Vehicle Driver Cognitive Load Measurement: Dataset and Baselines**.

Le papier introduit le dataset **CL-Drive**, dans lequel des signaux EEG, ECG, EDA et Gaze sont enregistrés pendant une tâche de conduite simulée. Les scores de charge cognitive sont collectés toutes les **10 secondes**. Après prétraitement, les signaux sont segmentés en fenêtres de 10 s, puis transformés en caractéristiques numériques pour entraîner des modèles de classification.

Dans ce TD, on se concentre uniquement sur les **features EEG**.

---

## Objectifs pédagogiques

À la fin du TD, vous devez être capables de :

1. expliquer pourquoi on transforme un signal EEG en vecteur de caractéristiques ;
2. distinguer les features temporelles, fréquentielles et non linéaires ;
3. expliquer le principe de la densité spectrale de puissance, ou PSD ;
4. calculer des puissances par bande EEG ;
5. interpréter l'entropie spectrale ;
6. expliquer les paramètres de Hjorth ;
7. comprendre le principe de la complexité de Lempel-Ziv ;
8. comprendre l'idée de la dimension fractale de Higuchi ;
9. structurer une fonction complète d'extraction de features EEG ;
10. préparer une matrice de features pour la classification.

---

## Features EEG ciblées

Le papier regroupe les features EEG suivantes :

| Famille | Features |
|---|---|
| PSD | puissance absolue, moyenne, maximale, minimale et médiane |
| Entropie spectrale | entropie calculée à partir de la PSD normalisée |
| Hjorth | mobility et complexity |
| Lempel-Ziv | complexité d'une séquence binarisée |
| Higuchi | dimension fractale |
| Statistiques temporelles | moyenne, minimum, maximum, médiane, variance, écart-type |

Dans ce TD, on adopte une version complète :

- PSD calculée dans les cinq bandes EEG : delta, theta, alpha, beta, gamma ;
- entropie spectrale calculée dans les cinq bandes ;
- features non linéaires calculées sur le segment temporel ;
- statistiques temporelles calculées sur le segment.

On obtient donc :

$$
5 \text{ bandes} \times 5 \text{ descripteurs PSD} = 25
$$

$$
5 \text{ entropies spectrales} = 5
$$

$$
2 \text{ Hjorth} + 1 \text{ Lempel-Ziv} + 1 \text{ Higuchi} + 6 \text{ statistiques} = 10
$$

Soit au total :

$$
25 + 5 + 10 = 40 \text{ features par canal EEG}
$$

## Questions de compréhension

### Question 1

Pourquoi ne donne-t-on pas directement le signal EEG brut à un classifieur classique comme LDA, SVM ou Random Forest ?

### Réponse 

Le signal EEG brut est très bruité, de grande dimension (ex: 2560 échantillons pour 10s), et non-stationnaire. Les classifieurs classiques souffrent du « fléau de la dimension » et peinent à y trouver des motifs pertinents sans extraction préalable de caractéristiques.

### Question 2

Pourquoi les features fréquentielles sont-elles particulièrement importantes en EEG ?

### Réponse 

L'activité cérébrale est caractérisée par des oscillations à des fréquences spécifiques (bandes delta, theta, alpha, beta, gamma). La puissance dans ces bandes est fortement corrélée aux états cognitifs (relaxation, charge mentale, attention), ce qui en fait des features discriminantes essentielles.

### Question 3

Pourquoi faut-il calculer les features séparément sur chaque canal EEG ?

### Réponse 

Différentes régions du cerveau gèrent différentes fonctions cognitives (ex: le cortex frontal est très impliqué dans la réflexion). Calculer les features par canal préserve cette information spatiale.

## 1. Bandes fréquentielles EEG

Les signaux EEG sont souvent analysés par bandes de fréquence.

| Bande | Intervalle utilisé dans ce TD | Interprétation générale |
|---|---:|---|
| Delta | 0.5–4 Hz | activité lente |
| Theta | 4–8 Hz | attention, mémoire de travail, somnolence selon contexte |
| Alpha | 8–12 Hz | relaxation, inhibition, yeux fermés |
| Beta | 12–30 Hz | activité mentale, attention, activité motrice |
| Gamma | 30–75 Hz | activité rapide, intégration, mais sensible aux artefacts musculaires |

Dans le papier, la bande gamma va jusqu'à 75 Hz. 

### Question 4

Pourquoi peut-on limiter la bande gamma à 45 Hz dans certaines implémentations ?

### Réponse 

Les hautes fréquences (au-delà de 45 Hz) sont souvent fortement polluées par les artefacts musculaires (EMG) et le bruit du réseau électrique (50 ou 60 Hz). Limiter la bande permet d'éviter de capter ce bruit.

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import welch
from scipy.stats import entropy

FS = 256
WINDOW_SEC = 10
WINDOW_SAMPLES = FS * WINDOW_SEC

EEG_BANDS = {
    "delta": (0.5, 4),
    "theta": (4, 8),
    "alpha": (8, 12),
    "beta": (12, 31),
    "gamma": (31, 75),
}

print("Nombre d'échantillons par fenêtre :", WINDOW_SAMPLES)
print("Bandes EEG :", EEG_BANDS)

In [ ]:
def generate_synthetic_signals(fs=FS, duration=10, random_state=0):
    rng = np.random.default_rng(random_state)
    t = np.arange(0, duration, 1/fs)
    signals = {
        "delta_2Hz": np.sin(2*np.pi*2*t),
        "alpha_10Hz": np.sin(2*np.pi*10*t),
        "beta_20Hz": np.sin(2*np.pi*20*t),
        "white_noise": rng.normal(0, 1, size=len(t)),
        "mixed": 0.8*np.sin(2*np.pi*6*t) + 0.5*np.sin(2*np.pi*10*t) + 0.2*rng.normal(0, 1, size=len(t)),
    }
    return t, signals

t, synthetic_signals = generate_synthetic_signals()

### Questions

1. Quelle bande doit dominer pour un signal sinusoïdal à 10 Hz ?
2. Pourquoi la PSD est-elle plus stable avec Welch qu'avec un simple spectre FFT ?
3. Que se passe-t-il si la bande sélectionnée ne contient aucune fréquence ?

### Réponses 

1. La bande alpha (8-12 Hz).
2. La méthode de Welch divise le signal en fenêtres, calcule la PSD pour chacune et fait la moyenne. Cela réduit fortement la variance de l'estimation au prix d'une perte de résolution en fréquence.
3. Le calcul de la puissance retournera 0 ou générera un `NaN` (ex: calcul de la moyenne sur un tableau vide).

In [ ]:
def compute_psd_band_features(signal, fs=FS, bands=EEG_BANDS):
    features = {}
    nperseg = min(len(signal), fs * 2)
    freqs, psd = welch(signal, fs=fs, nperseg=nperseg)
    
    for band_name, (f_min, f_max) in bands.items():
        idx = np.logical_and(freqs >= f_min, freqs < f_max)
        band_psd = psd[idx]
        
        if len(band_psd) > 0:
            features[f"psd_{band_name}_sum"] = np.sum(band_psd)
            features[f"psd_{band_name}_mean"] = np.mean(band_psd)
            features[f"psd_{band_name}_max"] = np.max(band_psd)
            features[f"psd_{band_name}_min"] = np.min(band_psd)
            features[f"psd_{band_name}_median"] = np.median(band_psd)
        else:
            features[f"psd_{band_name}_sum"] = 0.0
            features[f"psd_{band_name}_mean"] = 0.0
            features[f"psd_{band_name}_max"] = 0.0
            features[f"psd_{band_name}_min"] = 0.0
            features[f"psd_{band_name}_median"] = 0.0
            
    return features

### Questions

1. Pourquoi faut-il normaliser la PSD avant de calculer l'entropie ?
2. Quel signal devrait avoir l'entropie spectrale la plus élevée : une sinusoïde pure ou un bruit blanc ?
3. Pourquoi l'entropie spectrale peut-elle être utile pour caractériser la complexité d'un EEG ?

### Réponses 

1. L'entropie de Shannon calcule l'incertitude d'une distribution de probabilités, il faut donc que la somme des valeurs vale 1.
2. Le bruit blanc, car son énergie est distribuée uniformément sur toutes les fréquences.
3. Lors d'une charge cognitive complexe, le cerveau s'active de manière plus diversifiée et chaotique, ce qui augmente l'entropie, par rapport à un état de repos souvent dominé par la seule onde alpha.

In [ ]:
def compute_spectral_entropy_bands(signal, fs=FS, bands=EEG_BANDS):
    features = {}
    nperseg = min(len(signal), fs * 2)
    freqs, psd = welch(signal, fs=fs, nperseg=nperseg)
    
    for band_name, (f_min, f_max) in bands.items():
        idx = np.logical_and(freqs >= f_min, freqs < f_max)
        band_psd = psd[idx]
        
        if len(band_psd) > 0 and np.sum(band_psd) > 0:
            psd_norm = band_psd / np.sum(band_psd)
            features[f"entropy_{band_name}"] = entropy(psd_norm)
        else:
            features[f"entropy_{band_name}"] = 0.0
            
    return features

### Questions

1. Que vaut approximativement la variance d'un signal constant ?
2. Pourquoi faut-il gérer le cas où `Var(x)=0` ?
3. Entre une sinusoïde lisse et un bruit blanc, lequel devrait avoir une complexity plus élevée ?

### Réponses 

1. Zéro.
2. Pour éviter les erreurs de division par zéro lors du calcul de la mobilité et de la complexité.
3. Le bruit blanc, car il contient des variations rapides et brutales entre le signal et sa dérivée.

In [ ]:
def compute_hjorth(signal):
    features = {}
    var_x = np.var(signal)
    dx = np.diff(signal)
    var_dx = np.var(dx)
    ddx = np.diff(dx)
    var_ddx = np.var(ddx)
    
    if var_x == 0:
        features["hjorth_mobility"] = 0.0
        features["hjorth_complexity"] = 0.0
        return features
        
    mobility_x = np.sqrt(var_dx / var_x)
    features["hjorth_mobility"] = mobility_x
    
    if var_dx == 0 or mobility_x == 0:
        features["hjorth_complexity"] = 0.0
    else:
        mobility_dx = np.sqrt(var_ddx / var_dx)
        features["hjorth_complexity"] = mobility_dx / mobility_x
        
    return features

### Questions

1. Pourquoi faut-il binariser le signal avant de calculer Lempel-Ziv ?
2. Pourquoi la médiane est-elle un seuil intéressant ?
3. Quel signal devrait avoir une complexité plus élevée : une sinusoïde pure ou un bruit blanc ?
4. Pourquoi normaliser la complexité par la longueur de la séquence ?

### Réponses 

1. L'algorithme original de Lempel-Ziv compte l'apparition de nouveaux motifs dans une chaîne de symboles discrets (généralement binaires).
2. La médiane est très robuste face aux valeurs extrêmes (artefacts) comparée à la moyenne.
3. Le bruit blanc produira beaucoup plus de motifs uniques et imprévisibles qu'une sinusoïde.
4. Pour permettre une comparaison juste de la complexité entre des signaux de différentes longueurs.

In [ ]:
def lempel_ziv_complexity(signal):
    median_val = np.median(signal)
    b = (signal > median_val).astype(int)
    s = "".join(map(str, b))
    
    n = len(s)
    if n == 0:
        return {"lz_complexity": 0.0}
        
    complexity = 1
    prefix = s[0]
    prefix_len = 1
    
    while prefix_len < n:
        for j in range(prefix_len + 1, n + 1):
            sub = s[prefix_len:j]
            if sub not in s[:j-1]:
                complexity += 1
                prefix_len = j
                break
        else:
            break
            
    # Normalisation 
    b_norm = (n / np.log2(n)) if n > 1 else 1
    
    return {"lz_complexity": complexity / b_norm}

### Questions

1. Que cherche à mesurer la dimension fractale de Higuchi ?
2. Pourquoi un signal bruité peut-il avoir une dimension fractale plus élevée qu'une sinusoïde ?
3. Quel est le rôle du paramètre `kmax` ?
4. Pourquoi faut-il éviter de calculer un logarithme de zéro ?

### Réponses 

1. Elle évalue l'irrégularité et la complexité géométrique de la trajectoire du signal temporel.
2. Un signal bruité remplit davantage l'espace (plus tortueux et dentelé) qu'une onde lisse, ce qui augmente sa dimension fractale (proche de 2 en 1D).
3. Il définit le décalage (pas) maximal pour sous-échantillonner la courbe afin d'en mesurer la longueur à différentes échelles.
4. Le `log(0)` tend vers moins l'infini, générant des valeurs infinies (`-inf`) ou `NaN` qui bloquent le calcul de la pente.

In [ ]:
def higuchi_fd(signal, kmax=10):
    N = len(signal)
    if N < kmax:
        return {"higuchi_fd": 0.0}
        
    L = np.zeros(kmax)
    
    for k in range(1, kmax + 1):
        Lk = np.zeros(k)
        for m in range(k):
            idxs = np.arange(1, int(np.floor((N - m) / k)))
            diffs = np.abs(signal[m + idxs * k] - signal[m + (idxs - 1) * k])
            normalization = (N - 1) / (idxs[-1] * k) if len(idxs) > 0 else 0
            Lk[m] = np.sum(diffs) * normalization / k
        L[k - 1] = np.mean(Lk)
        
    # On évite le log(0)
    valid_k = [i for i in range(kmax) if L[i] > 0]
    if len(valid_k) < 2:
        return {"higuchi_fd": 0.0}
        
    x = np.log([1 / (i + 1) for i in valid_k])
    y = np.log([L[i] for i in valid_k])
    
    # Régression linéaire pour trouver la pente
    slope, _ = np.polyfit(x, y, 1)
    return {"higuchi_fd": slope}

In [ ]:
def compute_raw_features(signal):
    return {
        "raw_mean": np.mean(signal),
        "raw_min": np.min(signal),
        "raw_max": np.max(signal),
        "raw_median": np.median(signal),
        "raw_var": np.var(signal),
        "raw_std": np.std(signal),
    }

### Questions

1. Pourquoi la fonction doit-elle retourner un dictionnaire plutôt qu'une simple liste ?
2. Pourquoi est-il important de conserver des noms de colonnes explicites ?
3. Combien de features doit retourner la fonction pour un canal ?
4. Si on a 4 canaux et qu'on concatène toutes les features, combien de features obtient-on par segment ?

### Réponses 

1. Pour préserver les noms des features avec leurs valeurs associées, ce qui est très pratique pour la conversion en DataFrame pandas.
2. Pour assurer l'interprétabilité du modèle par la suite (ex: comprendre quelles features, ou quelles bandes de fréquences, influencent la classification).
3. 40 features.
4. 4 * 40 = 160 features.

In [ ]:
def extract_eeg_features(signal, fs=FS):
    signal = np.asarray(signal)
    
    # Remplacer les NaNs par 0 pour le calcul des features
    signal = np.nan_to_num(signal)
    
    features = {}
    features.update(compute_psd_band_features(signal, fs=fs))
    features.update(compute_spectral_entropy_bands(signal, fs=fs))
    features.update(compute_hjorth(signal))
    features.update(lempel_ziv_complexity(signal))
    features.update(higuchi_fd(signal))
    features.update(compute_raw_features(signal))
    
    return features

In [ ]:
print("\n--- Test sur les signaux synthétiques ---")
for name, sig in synthetic_signals.items():
    feats = extract_eeg_features(sig)
    print(f"\nSignal: {name}")
    # On n'affiche que quelques features clés pour vérifier
    print(f"  - PSD Alpha sum: {feats['psd_alpha_sum']:.4f}")
    print(f"  - PSD Beta sum: {feats['psd_beta_sum']:.4f}")
    print(f"  - Entropy Alpha: {feats['entropy_alpha']:.4f}")
    print(f"  - Hjorth Complexity: {feats['hjorth_complexity']:.4f}")
    print(f"  - Lempel-Ziv: {feats['lz_complexity']:.4f}")

## 12. Application aux signaux EEG du dataset CL-Drive

Après les tests pédagogiques, les mêmes fonctions doivent être appliquées aux signaux EEG prétraités.

### Hypothèse de structure des fichiers

On suppose que les fichiers EEG prétraités sont des fichiers CSV contenant :

- une colonne `Timestamp` ;
- une colonne par canal EEG, par exemple `AF7`, `AF8`, `TP9`, `TP10`.

Exemple de structure :

| Timestamp | AF7 | AF8 | TP9 | TP10 |
|---:|---:|---:|---:|---:|
| 0.000 | ... | ... | ... | ... |
| 0.004 | ... | ... | ... | ... |

### Algorithme d'extraction sur un fichier

1. lire le fichier CSV avec `pd.read_csv` ;
2. identifier les colonnes EEG ;
3. découper le signal en fenêtres de 10 secondes ;
4. pour chaque fenêtre :
   - extraire les 2560 échantillons ;
   - pour chaque canal, calculer les 40 features ;
   - stocker les métadonnées : sujet, fichier, fenêtre, temps début, temps fin, canal ;
5. construire un `DataFrame` ;
6. sauvegarder le résultat en CSV.

### Question

Pourquoi faut-il conserver les colonnes `sujet`, `scénario`, `fenêtre`, `canal`, `temps début` et `temps fin` avec les features ?

### Réponse

Ces métadonnées sont cruciales pour lier les features aux labels (charge cognitive évaluée toutes les 10s), mais aussi pour structurer la validation croisée (ex: Leave-One-Subject-Out) afin de ne pas mélanger les données d'un même sujet dans l'entraînement et le test.

In [ ]:
def extract_features_from_dataframe(df, eeg_cols=['AF7', 'AF8', 'TP9', 'TP10'], window_samples=WINDOW_SAMPLES, fs=FS):
    all_features = []
    num_windows = len(df) // window_samples
    
    for i in range(num_windows):
        start_idx = i * window_samples
        end_idx = start_idx + window_samples
        window_df = df.iloc[start_idx:end_idx]
        
        window_feats = {
            "fenetre": i,
            "temps_debut": window_df['Timestamp'].iloc[0] if 'Timestamp' in window_df.columns else start_idx / fs,
            "temps_fin": window_df['Timestamp'].iloc[-1] if 'Timestamp' in window_df.columns else end_idx / fs,
        }
        
        for col in eeg_cols:
            if col in window_df.columns:
                sig = window_df[col].values
                chan_feats = extract_eeg_features(sig, fs=fs)
                # Préfixer les noms avec le canal
                for k, v in chan_feats.items():
                    window_feats[f"{col}_{k}"] = v
                    
        all_features.append(window_feats)
        
    return pd.DataFrame(all_features)

In [ ]:
def batch_extract_eeg_features(base_path="Data/EEG", output_path="Data/EEG_Features_10s", fs=FS, window_samples=WINDOW_SAMPLES):
    base_path = Path(base_path)
    output_path = Path(output_path)
    output_path.mkdir(parents=True, exist_ok=True)
    
    for subject_dir in base_path.iterdir():
        if not subject_dir.is_dir():
            continue
            
        subject_id = subject_dir.name
        print(f"Traitement du sujet: {subject_id}")
        
        subject_features = []
        
        for file in subject_dir.glob("filtered_*.csv"):
            scenario_id = file.stem.replace("filtered_", "")
            df = pd.read_csv(file)
            
            feats_df = extract_features_from_dataframe(df, window_samples=window_samples, fs=fs)
            feats_df.insert(0, 'scénario', scenario_id)
            feats_df.insert(0, 'sujet', subject_id)
            
            subject_features.append(feats_df)
            
        if subject_features:
            final_subject_df = pd.concat(subject_features, ignore_index=True)
            out_file = output_path / f"{subject_id}_features.csv"
            final_subject_df.to_csv(out_file, index=False)
            print(f"Sauvegardé -> {out_file}")

## 14. Vérifications qualité des features

Avant de passer à la classification, il faut vérifier la qualité de la matrice de features.

### Vérifications recommandées

1. nombre de lignes cohérent avec le nombre de fenêtres et de canaux ;
2. absence de valeurs manquantes ;
3. absence de valeurs infinies ;
4. ordre de grandeur plausible ;
5. nombre de features égal à 40 par canal ;
6. conservation des métadonnées utiles ;
7. possibilité d'associer ensuite chaque fenêtre à un label.

### Questions

1. Pourquoi des valeurs `NaN` peuvent-elles apparaître dans les features ?
2. Pourquoi des valeurs infinies peuvent-elles apparaître ?
3. Que doit-on faire si un segment contient trop de valeurs manquantes ?
4. Pourquoi faut-il éviter de normaliser les features avant la séparation train/test ?

### Réponses 

1. Si une division par zéro se produit (ex: variance nulle, puissance nulle avant calcul d'entropie), ou si la bande de fréquence sélectionnée est vide.
2. Pour des raisons similaires aux NaN, souvent liées à des divisions par zéro ou des logarithmes de zéro.
3. Il est préférable de le supprimer ou d'utiliser des techniques d'imputation robustes, plutôt que de fausser l'apprentissage du modèle.
4. Pour éviter la fuite de données (data leakage). Les statistiques de normalisation (moyenne, écart-type) doivent être calculées uniquement sur l'ensemble d'entraînement et appliquées ensuite à l'ensemble de test.

In [ ]:
def check_features_quality(df):
    print("--- Contrôle Qualité des Features ---")
    print(f"Nombre de fenêtres (lignes) : {len(df)}")
    print(f"Nombre total de colonnes : {len(df.columns)}")
    
    # NaNs
    nan_counts = df.isna().sum()
    cols_with_nans = nan_counts[nan_counts > 0]
    if len(cols_with_nans) > 0:
        print(f"ALERTE: {len(cols_with_nans)} colonnes contiennent des NaNs.")
        print(cols_with_nans)
    else:
        print("OK: Aucun NaN détecté.")
        
    # Infinis
    inf_counts = np.isinf(df.select_dtypes(include=np.number)).sum()
    cols_with_infs = inf_counts[inf_counts > 0]
    if len(cols_with_infs) > 0:
        print(f"ALERTE: {len(cols_with_infs)} colonnes contiennent des valeurs infinies.")
        print(cols_with_infs)
    else:
        print("OK: Aucune valeur infinie détectée.")